In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from rapidfuzz import process, fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler, DamerauLevenshtein

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,remove_diacritics


In [2]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")

In [3]:
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)


what do I need?
- prep strings

- strings to compare: 
  - last name from variants
  - last name from unmatched

- things to consider
  - if "is_org", use whole string
  - 

In [4]:
var_dict = {}
name_list = []

org_keywords = ["stiftung", "archiv", "gesellschaft", "forum", "gemeinde", "sammlung", "institut", "museum", "kultur", "kuratorium", "verlag", "ministerium", "akademie", "trust", "verein", "organisation", "vereinigung", "universität", "bibliothek", "stadt", "staatlich", "werkstatt", "gemeinderat"]

for variant in variants:
    person_id = variant["person_id"]
    variant_normalised = variant["variant_normalised"]

    is_org = False
    if any(keyword in variant_normalised for keyword in org_keywords):
        is_org = True
        last = variant_normalised
        first = ""

    if not is_org:
        if ", " in variant_normalised:
            last, first = variant_normalised.split(", ", 1)
            # rprint(f"comma: {last, first}")

        else:
            split = variant_normalised.rsplit(" ", 1)
            if len(split) < 2:
                last = variant_normalised
                first = ""
                # rprint(f"only one found: {last}")

            else:
                first, last = split
                # rprint(f"first, lst split: {first, last}")
    # rprint(f"last, first: {last, first}")

    if not first:
        name_string = last
        last_string = last
    else:
        name_string = last + ", " + first
        last_string = last + ", " + first[:3]


    if name_string in var_dict:
        var_dict[name_string] = {
            "person_id": person_id,
            "last": last,
            "first": first,
            # "last_string": last_string
        }
    else:
        var_dict[name_string] = {
            "person_id": person_id,
            "last": last,
            "first": first,
            # "last_string": last_string
        }

# rprint(var_dict)
choices = list(var_dict.keys())
# last_list = [v["last_string"] for v in var_dict.values()]
# choices = last_list
# rprint(choices[:20])


In [5]:
# rprint(unmatched[:1])
# rprint(dict(list(unmatched.items())[:3]))
unmatched_dict = {}

for entry in unmatched:
    composite_id = entry["composite_id"]
    last = entry["last"]
    last_norm = normalise_text(last)
    first = entry["first"]
    first_norm = normalise_text(first)
    single = entry["single"]
    single_norm = normalise_text(single)
    is_org = entry["is_organisation"]

    if not last and single:
        first_norm = ""
        last_norm = single_norm

    if not first:
        name_string = last_norm

    else:
        name_string = last_norm + ", " + first_norm

    if name_string not in unmatched_dict:
        unmatched_dict[name_string] = {
            "composite_id": [composite_id],
            "display_name": entry["display_name"],
            "last_norm": last_norm,
            "family_name": last,
            "first_norm": first_norm,
            "given_names": first,
            "single_norm": single_norm,
            "single_name": single,
            "person_id": None,
            "match_found": None,
            "original_entry": entry["original_entry"],
            "notes": entry["verification_notes"]
        }
    else:
        unmatched_dict[name_string]["composite_id"].append(composite_id)

unmatched_list = list(unmatched_dict.keys())
# unmatched_last = [u["last_string"] for u in unmatched_dict.values()]
# unmatched_list = unmatched_last
rprint(unmatched_list[:3])

['rommel, otto', 'stigler-fuchs, margarete von', 'renolder, klemens']

In [6]:
results = {}


for name_string in unmatched_list:

    # top_matches = process.extract(name_string, choices, scorer=fuzz.ratio, limit=4)
    matches_all = process.extract(name_string, choices, scorer=JaroWinkler.normalized_similarity, limit=4)
    matches_info = [(name_string, score*100, var_dict[name_string]) for name_string, score, _ in matches_all]

    results[name_string] = matches_info

    # rprint(top_matches)
# rprint(dict(list(results.items())[:4]))

In [7]:
matched = {}
to_check = {}
nope = {}

matched_count = 0
to_check_count = 0
nope_count = 0

for name_string, matches in results.items():
# for name_string, matches in list(results.items())[:30]:
    info = unmatched_dict[name_string]
    composite_id = info["composite_id"]
    u_last = remove_diacritics(info["last_norm"])
    u_first = remove_diacritics(info["first_norm"])
    u_single = remove_diacritics(info["first_norm"])
    u_person_id = info["person_id"]
    u_match_found = info["match_found"]


    all_pids = []
    all_pids = [i[2]["person_id"] for i in matches]

    best_score = matches[0][1]


    if best_score >= 99:
        matched_count += 1
        matched[name_string] = {
            "match_found": matches[0],
            "u_person_id": matches[0][2]["person_id"],
            "u_match_found": True,
            "composite_id": composite_id
        }

    if best_score <= 99:
        points = 0
        if best_score >= 96:
            points += 2
        elif best_score >= 91:
            points += 1

        points += len(all_pids) - len(set(all_pids))

        matching_lasts = [m for m in matches if remove_diacritics(m[2]["last"]) == remove_diacritics(u_last)]
        # rprint(f"{name_string} → u_last={u_last!r}, matching_lasts={matching_lasts}")

        if len(matching_lasts) >= 1:
            points += 1

            initial_match = [i for i in matching_lasts if remove_diacritics(i[2]["first"][:2]) == remove_diacritics(u_first[:2])]
            if initial_match:
                points += 2

                even_more_matched = [e for e in initial_match if remove_diacritics(e[2]["first"][:4]) == remove_diacritics(u_first[:4])]
                if even_more_matched:
                    points += 2

        if points >= 5:
            matched_count += 1
            matched[name_string] = {
                "match_found": matches[0],
                "u_person_id": matches[0][2]["person_id"],
                "u_match_found": True,
                "composite_id": composite_id,
                "points": points
            }
        elif points >= 3:
            to_check_count += 1
            to_check[name_string] = {
                "points": points,
                "info": info,
                "matches": matches
            }
        else:
            nope_count += 1
            nope[name_string] = {
                "points": points,
                "info": info,
                "matches": matches
            }

rprint(f"matched count: {matched_count}")
rprint(f"to check count: {to_check_count}")
rprint(f"nope count: {nope_count}")


with open("matched.json", "w") as f:
    json.dump(matched, f, ensure_ascii=False, indent=2)

# with open("to_check.json", "w") as f:
#     json.dump(to_check, f, ensure_ascii=False, indent=2)

# with open("nope.json", "w") as f:
#     json.dump(nope, f, ensure_ascii=False, indent=2)


matched count: 324

to check count: 37

nope count: 1239